# UTS II4013 Data Analytics
## Personalized Data Auditor & Decision Analyst

**Nama:** Sebastian Albern Nugroho  
**NIM:**  18223074  
**Kelas:**  01  

### Pernyataan Keaslian
Saya menyatakan bahwa seluruh analisis pada notebook ini dikerjakan secara mandiri berdasarkan dataset yang dihasilkan dari NIM saya sendiri. Setiap keputusan analitik dijelaskan dengan alasan yang dapat saya pertanggungjawabkan.

## A. Tujuan Analisis

Jelaskan secara singkat:
1. Apa tujuan analisis Anda?
2. Apa target variabel yang akan dipelajari?
3. Keputusan apa yang ingin didukung oleh analisis ini?

**Tulis jawaban Anda di sini**

## B. Import Library
Jalankan sel berikut sebelum memulai analisis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (8, 5)

## C. Membuat Dataset Personal

Tuliskan NIM Anda dan jalankan generator dataset.  
Pastikan dataset yang dianalisis berasal dari NIM Anda sendiri.

In [2]:
# Isi dengan NIM masing-masing
nim = "18223074"

In [3]:
def generate_dataset(nim, n_samples=500):
    np.random.seed(int(nim[-3:]))

    noise_level = int(nim[1])       # digit ke-2
    missing_pattern = int(nim[3])   # digit ke-4
    bias_factor = int(nim[-1])      # digit terakhir

    timestamp = pd.date_range(start="2024-01-01", periods=n_samples, freq="h")

    temperature = 25 + 5 * np.sin(np.linspace(0, 10, n_samples)) + np.random.randn(n_samples)
    humidity = 60 + 10 * np.cos(np.linspace(0, 5, n_samples)) + np.random.randn(n_samples)

    student_count = np.random.randint(50, 500, n_samples)
    device_usage = student_count * np.random.uniform(0.5, 1.5, n_samples)

    day_type = np.where(pd.Series(timestamp).dt.dayofweek < 5, "weekday", "weekend")

    energy = (
        0.5 * temperature +
        0.3 * humidity +
        0.01 * student_count +
        0.02 * device_usage +
        bias_factor * 2
    )

    # Komponen non-linear
    energy += 5 * np.sin(student_count / 100)

    # Noise berdasarkan NIM
    energy += np.random.randn(n_samples) * noise_level

    df = pd.DataFrame({
        "timestamp": timestamp,
        "temperature": temperature,
        "humidity": humidity,
        "student_count": student_count,
        "device_usage": device_usage,
        "day_type": day_type,
        "energy_consumption": energy
    })

    # Inject missing values berdasarkan pola NIM
    if missing_pattern % 3 == 0:
        df.loc[df.sample(frac=0.1, random_state=1).index, "temperature"] = np.nan
    elif missing_pattern % 3 == 1:
        df.loc[df.sample(frac=0.1, random_state=1).index, "humidity"] = np.nan
    else:
        df.loc[df.sample(frac=0.1, random_state=1).index, "student_count"] = np.nan

    # Inject outliers pada target
    outlier_idx = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
    df.loc[outlier_idx, "energy_consumption"] *= (1 + bias_factor / 5)

    return df

In [4]:
df = generate_dataset(nim)
df.head()

,timestamp,temperature,humidity,student_count,device_usage,day_type,energy_consumption
0,2024-01-01 00:00:00,25.608345,70.923290,143.0,114.438115,weekday,44.166187
1,2024-01-01 01:00:00,24.463520,71.236029,259.0,345.715544,weekday,41.337741
2,2024-01-01 02:00:00,24.791470,70.008518,258.0,167.973013,weekday,51.424468
3,2024-01-01 03:00:00,26.152952,70.509952,499.0,730.609622,weekday,49.715757
4,2024-01-01 04:00:00,26.137910,71.091978,NaN,438.737058,weekday,50.699934


In [5]:
df.shape

(500, 7)

In [6]:
# Simpan dataset personal agar mudah dilampirkan saat pengumpulan
output_csv = f"dataset_UTS_{nim}.csv"
df.to_csv(output_csv, index=False)
print(f"Dataset tersimpan sebagai: {output_csv}")

Dataset tersimpan sebagai: dataset_UTS_18223074.csv


## D. Data Understanding

Lakukan pemeriksaan awal terhadap dataset:
1. Ukuran data
2. Nama variabel
3. Tipe data
4. Ringkasan statistik
5. Dugaan awal masalah kualitas data

**Tulis interpretasi Anda setelah kode dijalankan.**

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isna().sum()

### Interpretasi Awal
Jawab pertanyaan berikut:
- Variabel mana yang numerik dan kategorikal?
- Apakah ada missing value?
- Apakah ada nilai yang tampak tidak wajar?
- Apakah target variabel tampak cocok untuk regresi?

**Tulis jawaban Anda di sini**

## E. Audit Kualitas Data

Lakukan audit terhadap:
1. Missing values
2. Outliers
3. Ketidakkonsistenan format / tipe data
4. Potensi bias

In [ ]:
# Persentase missing value
missing_percent = df.isna().mean() * 100
missing_percent.sort_values(ascending=False)

In [ ]:
def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask.sum(), lower, upper, mask

outlier_summary = []
for col in ["temperature", "humidity", "student_count", "device_usage", "energy_consumption"]:
    count, lower, upper, mask = detect_outliers_iqr(df[col].dropna())
    outlier_summary.append({
        "column": col,
        "outlier_count": int(count),
        "lower_bound": float(lower),
        "upper_bound": float(upper)
    })

pd.DataFrame(outlier_summary)

### Hasil Audit
Isi uraian singkat:
- Variabel bermasalah:
- Jenis masalah:
- Dampak potensial terhadap analisis:

**Tulis jawaban Anda di sini**

## F. Penanganan Missing Value

Pilih satu metode imputasi yang menurut Anda paling tepat, misalnya:
- Mean
- Median
- Interpolation
- Metode lain

Jangan hanya menjalankan metode. Jelaskan:
1. Mengapa metode itu dipilih?
2. Mengapa metode lain tidak dipilih?

In [ ]:
df_clean = df.copy()

# Contoh dasar: imputasi median untuk variabel numerik
for col in ["temperature", "humidity", "student_count"]:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean.isna().sum()

### Justifikasi Missing Value Handling
**Tulis jawaban Anda di sini**

## G. Outlier Handling dan Transformasi

Tentukan apakah outlier:
- Dipertahankan
- Dibatasi
- Dihapus
- Ditangani dengan model robust

Lalu tentukan apakah data perlu scaling atau transformasi.

In [ ]:
# Contoh winsorization sederhana pada target
q1 = df_clean["energy_consumption"].quantile(0.25)
q3 = df_clean["energy_consumption"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_clean["energy_capped"] = df_clean["energy_consumption"].clip(lower=lower, upper=upper)
df_clean["target"] = df_clean["energy_capped"]

print("Lower bound:", lower)
print("Upper bound:", upper)

### Justifikasi Outlier dan Transformasi
- Apakah outlier dihapus atau dipertahankan?
- Mengapa?
- Apa dampaknya terhadap model?

**Tulis jawaban Anda di sini**

## H. Visualisasi Data

Buat minimal:
1. Histogram target
2. Scatter plot fitur vs target
3. Visualisasi hubungan dua fitur

In [ ]:
plt.hist(df_clean["target"], bins=30)
plt.title("Distribusi Target")
plt.xlabel("Target")
plt.ylabel("Frekuensi")
plt.show()

In [ ]:
plt.scatter(df_clean["temperature"], df_clean["target"])
plt.title("Temperature vs Target")
plt.xlabel("Temperature")
plt.ylabel("Target")
plt.show()

In [ ]:
plt.scatter(df_clean["student_count"], df_clean["device_usage"])
plt.title("Student Count vs Device Usage")
plt.xlabel("Student Count")
plt.ylabel("Device Usage")
plt.show()

### Interpretasi Visualisasi
Jawab:
- Apakah distribusi target simetris, skewed, atau memiliki ekor panjang?
- Fitur mana yang tampak memiliki hubungan kuat dengan target?
- Apakah ada pola non-linear?
- Apakah ada indikasi outlier atau cluster?

**Tulis jawaban Anda di sini**

## I. Encoding Variabel Kategorikal

Jika variabel kategorikal digunakan dalam model, lakukan encoding yang sesuai.

In [ ]:
df_model = df_clean.copy()
df_model = pd.get_dummies(df_model, columns=["day_type"], drop_first=True)
df_model.head()

## J. Analisis Korelasi

Lakukan:
1. Korelasi Pearson antar variabel numerik
2. Interpretasikan hasilnya
3. Jelaskan keterbatasannya

In [ ]:
corr_cols = ["temperature", "humidity", "student_count", "device_usage", "target"]
corr_matrix = df_model[corr_cols].corr()
corr_matrix

In [ ]:
plt.imshow(corr_matrix, interpolation="nearest")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar()
plt.title("Correlation Matrix")
plt.show()

### Interpretasi Korelasi
Jawab:
- Fitur mana yang paling berkorelasi dengan target?
- Apakah korelasinya positif atau negatif?
- Apakah ada pasangan fitur yang saling berkorelasi tinggi?
- Apakah korelasi cukup untuk menyimpulkan sebab-akibat?

**Tulis jawaban Anda di sini**

## K. Persiapan Data untuk Pemodelan Regresi

Pilih fitur input dan target.  
Jelaskan mengapa fitur tertentu digunakan atau tidak digunakan.

In [ ]:
feature_cols = ["temperature", "humidity", "student_count", "device_usage", "day_type_weekend"]
X = df_model[feature_cols]
y = df_model["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test :", X_test.shape)

### Justifikasi Pemilihan Fitur
**Tulis jawaban Anda di sini**

## L. Model 1 – Linear Regression

Bangun model baseline dengan regresi linear.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred_lin = lin_reg.predict(X_test)

In [ ]:
mae_lin = mean_absolute_error(y_test, y_pred_lin)
mse_lin = mean_squared_error(y_test, y_pred_lin)
rmse_lin = np.sqrt(mse_lin)
r2_lin = r2_score(y_test, y_pred_lin)

print("Linear Regression Performance")
print("MAE :", mae_lin)
print("MSE :", mse_lin)
print("RMSE:", rmse_lin)
print("R2  :", r2_lin)

In [ ]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lin_reg.coef_
})
coef_df

### Interpretasi Model Linear
Jawab:
- Apa arti koefisien masing-masing fitur?
- Fitur mana yang paling dominan?
- Apakah tanda koefisien sesuai dugaan awal?
- Apakah performa model cukup baik?

**Tulis jawaban Anda di sini**

## M. Model 2 – Huber Regression

Bangun model robust untuk membandingkan sensitivitas terhadap outlier.

In [ ]:
huber = HuberRegressor()
huber.fit(X_train, y_train)

y_pred_huber = huber.predict(X_test)

In [ ]:
mae_huber = mean_absolute_error(y_test, y_pred_huber)
mse_huber = mean_squared_error(y_test, y_pred_huber)
rmse_huber = np.sqrt(mse_huber)
r2_huber = r2_score(y_test, y_pred_huber)

print("Huber Regression Performance")
print("MAE :", mae_huber)
print("MSE :", mse_huber)
print("RMSE:", rmse_huber)
print("R2  :", r2_huber)

### Perbandingan Model
Bandingkan Linear Regression dan Huber Regression:
- Model mana yang lebih baik?
- Mengapa?
- Apa kaitannya dengan keberadaan outlier?

**Tulis jawaban Anda di sini**

## N. Visualisasi Prediksi vs Aktual

In [ ]:
plt.scatter(y_test, y_pred_lin)
plt.title("Actual vs Predicted - Linear Regression")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
plt.scatter(y_test, y_pred_huber)
plt.title("Actual vs Predicted - Huber Regression")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

### Interpretasi Prediksi
- Apakah prediksi mengikuti nilai aktual?
- Di bagian mana model gagal?
- Apakah ada pola error tertentu?

**Tulis jawaban Anda di sini**

## O. Analisis Kritis dan Refleksi

Jawab secara argumentatif:

1. Apa tiga temuan utama dari analisis Anda?
2. Apa risiko jika hasil model langsung dipakai untuk pengambilan keputusan?
3. Apakah hubungan yang ditemukan bersifat korelasional atau kausal?
4. Apa keterbatasan dataset Anda?
5. Jika diberi kesempatan memperbaiki analisis, apa yang akan Anda lakukan?

**Tulis jawaban Anda di sini**

## P. Log Keputusan Analitik

Isi tabel berikut:

| Tahap | Keputusan | Alasan |
|------|-----------|--------|
| Missing value | ... | ... |
| Outlier | ... | ... |
| Transformasi | ... | ... |
| Fitur model | ... | ... |
| Model akhir | ... | ... |

Lengkapi tabel dengan keputusan Anda sendiri.

## Q. Kesimpulan

Tuliskan kesimpulan akhir dalam 1–2 paragraf:
- Apa yang berhasil ditemukan?
- Model mana yang paling layak?
- Apakah hasil cukup kuat untuk mendukung keputusan?

**Tulis jawaban Anda di sini**

## Ketentuan Penting
1. Tidak cukup hanya menampilkan output kode.
2. Setiap tahap wajib disertai interpretasi.
3. Setiap keputusan preprocessing dan modeling wajib diberi alasan.
4. Jawaban generik tanpa keterkaitan dengan hasil aktual notebook akan dinilai rendah.
5. Fokus utama penilaian adalah kualitas penalaran, bukan sekadar angka performa.